In [1]:
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F
torch.manual_seed(0)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = '/kaggle/input/datasets/arnavnagpure/cifar10/cifar10.pt'
d = torch.load(DATA_PATH, map_location=device)
print(device)

cuda


In [3]:
def char_fn(X, t: torch.Tensor):
    # Applies the characteristic function over a
    # range of different values of t for an X grouped by direction
    
    # Unsqueeze to group by direction and apply
    # all the t values to every direction
    tX = X.unsqueeze(1) * t.unsqueeze(1)   # (64, t, num points)

    # 𝜙_X(𝑡) = E[e^itX] = 1/N * Σe^itx
    return torch.exp(1j * tX).mean(2)


def epps_pulley(X):
    # EP = N∫|𝜙_X(𝑡) - 𝜙(t)|²w(t)dt
    N = X.size(1)
    
    # Range of t values for estimating the integral
    t = torch.linspace(-5, 5, 33, device=X.device)
    

    # Work out characteristic functions,
    # target cf is the same as the weight w(t)
    target_cf = w = torch.exp(-0.5 * t ** 2) # 𝜙(𝑡) 
    ecf = char_fn(X, t)                      # 𝜙_X(𝑡)

    
    err = (target_cf - ecf).abs().square().mul(w) # |𝜙_X(𝑡) - 𝜙(t)|²w(t)
    return N * torch.trapezoid(err, t, dim=1)


# SIGReg!!!!
def SIGReg(Z, dirs=64):
    # Generate 64 random unit vectors on hypersphere S^(N-1)
    A = torch.randn(dirs, Z.size(1), device=Z.device)
    A = A / torch.linalg.vector_norm(A, dim=1, keepdim=True)

    # For each direction project every point (Z) onto it to get
    # a line of points (which we check is normally distributed)
    X = A @ Z.T        # (64, num points)

    if X.abs().max() > 10:
        print(f"ERROR: {X.abs().max()}")

    EP = epps_pulley(X)
    return EP
    

In [4]:
# -- Helper functions --

# Plot view pairs in grid WIP
def plot_grid(views_list):
  w = h = 32
  fig = plt.figure(figsize=(7, 14))
  columns = len(views_list)
  rows = len(views_list[0])
  for i in range(columns*rows):
      img = views_list[i % columns][i // columns]
      fig.add_subplot(rows, columns, i+1)
      plt.imshow(img.cpu().permute(1,2,0), interpolation='nearest')
      plt.axis('off')
  plt.subplots_adjust(wspace=0, hspace=0)
  plt.show()


# Other helpers made by Claude
# (batch image transforms not supported by torchvision)
LUMA = torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)

def _rand(B, lo, hi, device):
    return torch.empty(B, 1, 1, 1, device=device).uniform_(lo, hi)

def _gray(x):                                  # (B,3,H,W) -> (B,3,H,W)
    return (x * LUMA.to(x.device)).sum(1, keepdim=True).expand_as(x)

def brightness_(x, lo=0.6, hi=1.4):
    x.mul_(_rand(x.size(0), lo, hi, x.device))
    return x

def contrast_(x, lo=0.6, hi=1.4):
    f = _rand(x.size(0), lo, hi, x.device)
    mean = _gray(x).mean((1, 2, 3), keepdim=True)   # scalar per image
    x.lerp_(mean, 1 - f)
    return x

def saturation_(x, lo=0.6, hi=1.4):
    f = _rand(x.size(0), lo, hi, x.device)
    x.lerp_(_gray(x), 1 - f)
    return x

def hue_(x, amount=0.1):
    # cheap approximation: rotate the two chroma axes about luma
    f = _rand(x.size(0), -amount, amount, x.device) * 2 * 3.141592653589793
    c, s = torch.cos(f), torch.sin(f)
    g = _gray(x)[:, :1]
    r, gr, b = x[:, 0:1] - g, x[:, 1:2] - g, x[:, 2:3] - g
    x[:, 0:1] = g + c * r - s * b
    x[:, 1:2] = g + gr                              # green anchored
    x[:, 2:3] = g + s * r + c * b
    return x

def hflip_(x, p=0.5):
    m = torch.rand(x.size(0), device=x.device) < p
    x[m] = x[m].flip(-1)
    return x

def greyscale_(x, p=0.2):
    m = torch.rand(x.size(0), device=x.device) < p
    x[m] = _gray(x[m])
    return x

def rrc(x, out=32, scale=(0.4, 1.0), ratio=(3/4, 4/3)):
    B, dev = x.size(0), x.device
    area = torch.empty(B, device=dev).uniform_(*scale)
    logr = torch.empty(B, device=dev).uniform_(math.log(ratio[0]), math.log(ratio[1]))
    ar = torch.exp(logr)
    w = (area * ar).sqrt().clamp(max=1.0)      # crop width as fraction of image
    h = (area / ar).sqrt().clamp(max=1.0)
    cx = torch.rand(B, device=dev) * (1 - w) * 2 - (1 - w)   # centre in [-1,1] coords
    cy = torch.rand(B, device=dev) * (1 - h) * 2 - (1 - h)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 2] = w, cx
    theta[:, 1, 1], theta[:, 1, 2] = h, cy
    grid = F.affine_grid(theta, (B, 3, out, out), align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', align_corners=False)

In [5]:
# Takes in a batch of images and returns a batch of tokenised images.
# Each image is split into patches and each patch
# is flattened and transformed into a 192dim token.
# Positional embeddings are then added on.
class Tokeniser(torch.nn.Module):

  def __init__(self, IMAGE_SIZE, OUT_DIMS, PATCH_SIZE):
    super().__init__()
    self.TOKEN_COUNT = (IMAGE_SIZE // PATCH_SIZE)**2
    self.patch_embedder = torch.nn.Conv2d(3, OUT_DIMS, PATCH_SIZE, stride=PATCH_SIZE)
    self.pos_embeddings = torch.nn.parameter.Parameter(torch.randn(self.TOKEN_COUNT, OUT_DIMS) * 0.03)

  def forward(self, X):
    # Convert from (N, 192, 8, 8) to (N, 64, 192)
    tokens = self.patch_embedder(X).flatten(2).transpose(1,2)
    tokens = tokens + self.pos_embeddings
    return tokens


class Attention(torch.nn.Module):

  def __init__(self, N_HEADS, DIMS):
    super().__init__()
    self.W_KQV = torch.nn.Linear(DIMS, DIMS*3, bias=False)
    self.W_O = torch.nn.Linear(DIMS, DIMS)
    self.N_HEADS = N_HEADS
    self.DIMS = DIMS

  def forward(self, X):
    # Compute KQV all at once, for every element in the batch
    KQV = self.W_KQV(X)

    # Split into separate K, Q and V
    K, Q, V = KQV.split(self.DIMS, dim=-1)

    # Split across heads into (N, Tokens, Heads, Elements)
    d_k = self.DIMS // self.N_HEADS
    shape = K.shape[:2] + (self.N_HEADS, d_k)

    K = K.reshape(shape)
    Q = Q.reshape(shape)
    V = V.reshape(shape)

    # Also permute to (N, Heads, Tokens, Elements) for matmuls
    K = K.permute((0,2,1,3))
    Q = Q.permute((0,2,1,3))
    V = V.permute((0,2,1,3))

    weights = Q @ K.mT
    weights = weights / math.sqrt(d_k)
    weights = F.softmax(weights, dim=-1)

    embeddings = weights @ V
    # (N, heads, tokens, head_size) e.g. (N, 6, 64, 32)
    embeddings_shape = shape[:2] + (self.DIMS, )
    # (N, 6, 64, 32) --> (N, 64, 6, 32) --> (N, 64, 192)
    embeddings = embeddings.transpose(1, 2).contiguous().reshape(embeddings_shape)
    embeddings = self.W_O(embeddings)

    return embeddings


class Transformer_Block(torch.nn.Module):

  def __init__(self, DIMS, N_HEADS):
    super().__init__()
    self.norm_a = torch.nn.LayerNorm(DIMS)
    self.norm_b = torch.nn.LayerNorm(DIMS)
    self.attention = Attention(N_HEADS, DIMS)
    self.MLP = torch.nn.Sequential(
        torch.nn.Linear(DIMS, 4 * DIMS),
        torch.nn.GELU(),
        torch.nn.Linear(4 * DIMS, DIMS)
    )

  def forward(self, X):
    X = X + self.attention(self.norm_a(X))
    X = X + self.MLP(self.norm_b(X))
    return X


class ViT(torch.nn.Module):

  def __init__(self, IMAGE_SIZE, PATCH_SIZE=4, DIMS=192, N_HEADS=6):
    super().__init__()
    self.tokeniser = Tokeniser(IMAGE_SIZE, DIMS, PATCH_SIZE)
    self.transformer_blocks = torch.nn.Sequential(
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS),
      Transformer_Block(DIMS, N_HEADS)
    )
    self.norm = torch.nn.LayerNorm(DIMS)

  def forward(self, X):
    tokens    = self.tokeniser(X)
    patch_embeddings = self.transformer_blocks(tokens)
    patch_embeddings = self.norm(patch_embeddings)
    image_embedding  = patch_embeddings.mean(dim=1)

    return image_embedding


In [6]:
def create_view(x):
  # Expects non standardised data in range [0,1]
  view = rrc(x)
  if random.random() < 0.8: # This is batch level not sample level - TODO?
    brightness_(view)
    contrast_(view)
    saturation_(view)
    hue_(view)
  hflip_(view)
  greyscale_(view)
  return view
  #return view.clamp_(0, 1)

In [7]:
# Collapse metric helper functions.
# Help determine if rep collapse is occurring.

# Calculates entropy from a probability distribution
def entropy(P):
  # Compute entopy letting p*log(p) = 0 when p = 0
  H = torch.where(P > 0, P * P.log(), 0.0)
  return -H.sum()

def effective_rank(Z):
  # Make mean zero
  Z = Z - Z.mean(dim=0)
  # Compute covariance matrix
  C = (Z.T @ Z) / len(Z)
  # Calculate eigenvalues (=variances) for directions which maximise
  # covariance i.e. directions that are principle for distribution
  eigenvalues = torch.linalg.eigvalsh(C)

  # Check for total collapse (~one point)
  if eigenvalues.sum() < 1E-8:
    return torch.tensor(1.0, device=Z.device)

  # Normalise to turn into a probability distribution
  P = eigenvalues / eigenvalues.sum()
  H = entropy(P)
  return torch.exp(H) # Effective rank

def mean_pairwise_cosine(Z, CENTERED=False, SAMPLE_SIZE=4096):
  if CENTERED:
    # Make mean zero / shift to origin
    Z = Z - Z.mean(dim=0)

  # Take small sample for performance
  idx = torch.randperm(len(Z))[:SAMPLE_SIZE]
  Z = Z[idx]

  # Check for total collapse (~one point)
  norms = torch.linalg.vector_norm(Z, dim=1)
  if (norms < 1E-16).all():
    return torch.tensor(1.0, device=Z.device)

  # Normalise
  Z = F.normalize(Z, dim=1)
  cosine_pairs = Z @ Z.T
  N = len(cosine_pairs)

  # Sum all pairs and subtract the diagonal (which is always 1)
  S = cosine_pairs.abs().sum() - N
  return S / (N**2 - N)

In [8]:
@torch.inference_mode()
def cache_embeddings(encoder, X, BATCH_SIZE=4096):
  encoder.eval()
  embeddings = [encoder(norm(X[i:i+BATCH_SIZE])) for i in range(0, len(X), BATCH_SIZE)]
  return torch.cat(embeddings)

In [9]:
@torch.inference_mode()
def report(embedder, X, epoch='Stats'):
  Z = cache_embeddings(embedder, X)

  erank = effective_rank(Z)
  mp_cos = mean_pairwise_cosine(Z)
  mp_cos_cent = mean_pairwise_cosine(Z, CENTERED=True)
  std = Z.std(dim=0)

  print(f'{epoch:3} | erank {erank:.2f} | cos {mp_cos:.3f} / {mp_cos_cent:.3f} | std {std.min():.2f} {std.median():.2f} {std.max():.2f}')

In [10]:
@torch.inference_mode()
def evaluate(model, X, Y):
  model.eval()

  N = len(Y)
  Y_pred = model(X)
  correct = (Y_pred.argmax(1) == Y).sum().item()

  return correct / N

In [11]:
# Shuffle and split train dataset into 45K/5K
# VAL IS NOT USED - 45K split still needed for comparison since
# baseline benchmarks were computed with 45K train data
data_perm = torch.randperm(50000)
train_idx = data_perm[:45_000]

# Load and set range to [0,1]
train_data_X, train_data_Y = d['xtr'][train_idx].float() / 255, d['ytr'][train_idx]
test_data_X, test_data_Y   = d['xte'].float() / 255,            d['yte']

# Use mean/std from train only
mean = train_data_X.mean((0, 2, 3), keepdim=True)
std  = train_data_X.std((0, 2, 3), keepdim=True)

def norm(x):
    return (x - mean) / std

# Standardise
#train_data_X = (train_data_X - mean) / std
#test_data_X  = (test_data_X  - mean) / std

In [12]:
BATCH_SIZE = 128
LAMBDA = 0.01
IMAGE_SIZE = train_data_X.shape[-1]
N = len(train_data_X)

embedder = ViT(32, 4, 192, 6).to(device)

# Not needed for JEPA but useful for testing
predictor = torch.nn.Sequential(
    torch.nn.Linear(192, 512),
    torch.nn.GELU(),
    torch.nn.Linear(512, 192)
)

optimiser = torch.optim.AdamW(embedder.parameters(), lr=4E-4)

In [18]:
for epoch in range(21, 60+1):
  embedder.train()
  perm = torch.randperm(N, device=device)
  loss_value = -1

  for i in range(0, N - N % BATCH_SIZE, BATCH_SIZE):
    idx = perm[i:i+BATCH_SIZE]
    X = train_data_X[idx]

    # Generate views by (applying transforms)
    views_a = create_view(X)
    views_b = create_view(X)
      
    # Embed the views 
    z_a = embedder(norm(views_a))
    z_b = embedder(norm(views_b))
    all_embeddings = torch.stack((z_a, z_b))
    
    centers = torch.stack((z_a, z_b)).mean(0)  # Find mean of every view pair 
    
    # Each pair should encode the same information so find the difference/dist sq
    # and try to minimise it
    sim_loss = (centers.unsqueeze(0) - all_embeddings).square().mean()
    # Ensure the embeddings still stay well distributed (as an isotropic Gaussian)
    sig_loss = (SIGReg(z_a).mean() + SIGReg(z_b).mean()) * 0.5
    loss = (1 - LAMBDA) * sim_loss + LAMBDA * sig_loss

    # Debug stats
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"{sim_loss.item():.4f} | {sig_loss.item():.4f}")

    if (i // BATCH_SIZE) % 500 == 0:
        g_sim = torch.autograd.grad(sim_loss, all_embeddings, retain_graph=True)[0].norm()
        g_sig = torch.autograd.grad(sig_loss, z_a, retain_graph=True)[0].norm()
        print(g_sim, g_sig)
        
        report(embedder, train_data_X[:8192])
        embedder.train()
      
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
      
    loss_value = loss.item()

    #print(sim_loss, sig_loss)
  if epoch % 5 == 0:
    torch.save(embedder.state_dict(), f'/kaggle/working/jepa_e{epoch}.pt')

  print(f'Loss for epoch {epoch:<2} | {loss_value:.4f}')

0.0420 | 6.5038
tensor(0.0018, device='cuda:0') tensor(0.2846, device='cuda:0')
Stats | erank 7.85 | cos 0.300 / 0.303 | std 0.28 0.81 2.51
0.0339 | 6.8358
0.0400 | 5.7304
0.0478 | 6.2895
0.0433 | 6.8057
0.0466 | 6.7740
0.0483 | 6.5093
0.0445 | 5.2426
0.0430 | 7.9168
0.0468 | 6.0170
0.0468 | 5.4936
0.0508 | 6.6271
0.0448 | 7.1755
0.0409 | 6.5192
0.0376 | 5.8260
0.0429 | 6.4783
0.0418 | 7.0974
0.0483 | 7.2319
0.0411 | 5.8721
0.0413 | 6.1179
0.0357 | 7.1342
0.0393 | 6.6735
0.0438 | 6.4214
0.0358 | 7.0177
0.0393 | 6.7369
0.0423 | 6.1301
0.0421 | 5.9786
0.0400 | 7.1856
0.0378 | 6.7303
0.0352 | 5.9243
0.0355 | 7.4966
0.0455 | 5.3793
0.0370 | 6.7192
0.0446 | 5.4967
0.0432 | 6.8956
0.0394 | 8.1647
Loss for epoch 21 | 0.1207
0.0360 | 6.0842
tensor(0.0017, device='cuda:0') tensor(0.2569, device='cuda:0')
Stats | erank 7.96 | cos 0.297 / 0.299 | std 0.29 0.79 2.27
0.0364 | 6.6167
0.0349 | 5.9401
0.0423 | 5.3227
0.0427 | 5.7184
0.0436 | 7.3395
0.0411 | 4.7209
0.0353 | 5.4702
0.0395 | 6.9958
0.044

KeyboardInterrupt: 

In [14]:
torch.save({'model': embedder.state_dict()},
           '/kaggle/working/jepa.pt')

Cleanup TODOs:
- Move all the helper functions out into another file
- Switch position embeddings to randomised
- Add augmentation for both CNN and ViT
- Add types and proper annotations for every function

In [15]:
# Creates, trains and tests a linear probe
def linear_probe(encoder, Xtr, Ytr, Xte, Yte, device='cpu', BATCH_SIZE=1024, EPOCHS=20):

  @torch.inference_mode()
  def cache_embeddings(encoder, X):
    encoder.eval()
    embeddings = [encoder(norm(X[i:i+BATCH_SIZE])) for i in range(0, len(X), BATCH_SIZE)]
    return torch.cat(embeddings)


  # Cache embeddings for training
  embeddings = cache_embeddings(encoder, Xtr)
  embedding_dims = len(embeddings[0])
  N_classes = Ytr.max().item() + 1
  N = len(Ytr)


  # Create the probe and prepare for training
  probe = torch.nn.Linear(embedding_dims, N_classes).to(device)
  loss_fn = torch.nn.CrossEntropyLoss().to(device)
  optimiser = torch.optim.AdamW(probe.parameters(), lr=1e-3)


  # Train probe
  for epoch in range(EPOCHS):
    probe.train()
    perm = torch.randperm(N, device=device)

    for i in range(0, N - N % BATCH_SIZE, BATCH_SIZE):
      idx = perm[i:i+BATCH_SIZE]
      batch_X, batch_Y = embeddings[idx], Ytr[idx]

      Y_pred = probe(batch_X)
      loss = loss_fn(Y_pred.float(), batch_Y)
      optimiser.zero_grad()
      loss.backward()
      optimiser.step()


  # Test probe
  X = cache_embeddings(encoder, Xte)
  acc = evaluate(probe, X, Yte)

  return probe, acc

In [16]:
probe, acc = linear_probe(embedder, train_data_X, train_data_Y, test_data_X, test_data_Y, device)
acc

0.4053

In [17]:
report(embedder, train_data_X)

Stats | erank 7.87 | cos 0.299 / 0.301 | std 0.28 0.80 2.50


In [19]:
import glob, re, json

results = []
for path in sorted(glob.glob('/kaggle/working/jepa_e*.pt'),
                   key=lambda p: int(re.search(r'e(\d+)', p).group(1))):
    ep = int(re.search(r'e(\d+)', path).group(1))
    m = ViT(32, 4, 192, 6).to(device)
    m.load_state_dict(torch.load(path, map_location=device))   # bare state_dict
    _, acc = linear_probe(m, train_data_X, train_data_Y, test_data_X, test_data_Y, device)
    Z = cache_embeddings(m, train_data_X)
    results.append({'epoch': ep, 'acc': acc,
                    'erank': effective_rank(Z).item(),
                    'cos': mean_pairwise_cosine(Z).item()})
    print(results[-1])

json.dump(results, open('/kaggle/working/probe_curve.json', 'w'))

{'epoch': 5, 'acc': 0.3629, 'erank': 5.904085636138916, 'cos': 0.3525623381137848}
{'epoch': 10, 'acc': 0.3851, 'erank': 6.005716800689697, 'cos': 0.34710919857025146}
{'epoch': 15, 'acc': 0.3986, 'erank': 6.142181396484375, 'cos': 0.3425039052963257}
{'epoch': 20, 'acc': 0.41, 'erank': 7.871391296386719, 'cos': 0.2993176281452179}
{'epoch': 25, 'acc': 0.427, 'erank': 8.105542182922363, 'cos': 0.29354971647262573}


First run
```
{'epoch': 5, 'acc': 0.3629, 'erank': 5.904085636138916, 'cos': 0.3525623381137848}
{'epoch': 10, 'acc': 0.3851, 'erank': 6.005716800689697, 'cos': 0.34710919857025146}
{'epoch': 15, 'acc': 0.3986, 'erank': 6.142181396484375, 'cos': 0.3425039052963257}
{'epoch': 20, 'acc': 0.41, 'erank': 7.871391296386719, 'cos': 0.2993176281452179}
{'epoch': 25, 'acc': 0.427, 'erank': 8.105542182922363, 'cos': 0.29354971647262573}
```
